# Gemma 2B-IT — Türkçe QLoRA fine-tune (Faz 3)

Bu notebook `google/gemma-2b-it` modelini Türkçe soru-cevap verisiyle **QLoRA** yöntemiyle eğitir ve adaptörü Hugging Face Hub'a yükler.

**Akış**

1. Kütüphaneler
2. Hugging Face girişi (Gemma gated model)
3. Temel modeli 4-bit yükle
4. Dataset: yükle, filtrele, Gemma şablonuna çevir
5. LoRA yapılandırması
6. Eğitim (3 epoch) + her epoch sonunda eval
7. Eğitim sonu kontrol
8. Adaptörü Hub'a yükle (private)

**Önemli kurallar**

- Temel model **`google/gemma-2b-it`** (Gemma 1). `gemma-2-2b-it` **kullanılmaz** — farklı mimari, projedeki MLC modeliyle uyuşmaz.
- GPU şart. **Colab:** Runtime → Change runtime type → T4 GPU. **Kaggle:** Settings → Accelerator → GPU T4 x2 ve Internet → On.
- Bu notebook yalnızca adaptör üretir; adaptörün MLC yığınına bağlanması ayrı aşamadır (Faz 5).

**2. tur değişiklikleri (1. turun sonuçlarına göre)**

| Sorun | Düzeltme |
|---|---|
| Cevaplar bitmiyor, içine `model` kelimesi sızıyor | Üretimde `<end_of_turn>` durdurucu token olarak eklendi |
| Model soruyu da üretmeyi öğreniyor | Completion-only loss: soru token'ları `-100` ile maskelendi |
| Çağrışımsal, geveze cevaplar | Sohbet satırı oranı 0.4 → 0.15 |
| 3. epoch'tan sonra `eval_loss` yükseliyor | 4 → 3 epoch, learning rate 2e-4 → 1e-4 |
| Colab kopunca her şey kayboluyor | Checkpoint'ler Google Drive'a yazılıyor |

## 1. Kütüphaneler

`peft` LoRA katmanlarını, `bitsandbytes` 4-bit nicelemeyi, `accelerate` ise GPU yerleşimini sağlar. Bu üçü olmadan 2B model ücretsiz T4'ün 15 GB belleğine eğitim için sığmaz.

**Bu hücreden sonra çekirdeği mutlaka yeniden başlatın** (Kaggle: Run → Restart session, Colab: Runtime → Restart session). Kurulum diskteki dosyaları değiştirir ama bellekteki eski modüller yerinde kalır; bu karışım `GemmaConfig must be a dataclass` gibi anlaşılmaz hatalara yol açar. Yeniden başlattıktan sonra bu hücreden itibaren sırayla devam edin.

Sürümler bilerek sabitlendi. `-U` ile her seferinde en yeni sürümü çekmek, `transformers` ve `huggingface_hub` arasında uyumsuzluk riski taşıyor.

**Kaggle kullanıyorsanız:** sağ paneldeki Settings bölümünde `Accelerator → GPU T4 x2` ve `Internet → On` olmalı. İnternet kapalıysa ne pip kurulumu ne de model indirmesi çalışır.

In [ ]:
import os

# Kaggle iki T4 verir. 4-bit + LoRA egitimini tek GPU'da tutmak, modelin iki
# karta bolunmesinden dogan hatalari onler. Colab'da zaten tek GPU vardir.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Surumler sabitlendi. "-U" ile her calistirmada en yeni surum cekmek,
# transformers ile huggingface_hub arasinda uyumsuzluga yol acabiliyor
# (ornek hata: "Class 'GemmaConfig' must be a dataclass before applying @strict").
!pip install -q \
    "transformers==5.14.1" \
    "huggingface_hub==1.26.0" \
    "datasets==5.0.1" \
    "accelerate==1.14.0" \
    "peft==0.20.0" \
    "bitsandbytes==0.50.0"

import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — Runtime > Change runtime type > GPU")

## 2. Hugging Face girişi

`google/gemma-2b-it` **gated** bir modeldir: indirebilmek için önce model sayfasında lisansı kabul etmeniz gerekir.

1. https://huggingface.co/google/gemma-2b-it adresinde lisansı kabul edin.
2. https://huggingface.co/settings/tokens adresinden **write** yetkili bir token oluşturun (adaptörü Hub'a yükleyeceğiz).
3. Token'ı platformun gizli anahtar deposuna `HF_TOKEN` adıyla ekleyin:
   - **Colab:** sol menü → anahtar simgesi (Secrets) → ekleyin, notebook erişimini açın.
   - **Kaggle:** sağ panel → Add-ons → Secrets → ekleyin, notebook'a bağlayın.
   - Hiçbiri yoksa hücre token'ı gizli girdi olarak soracaktır.

Token'ı hücreye düz metin olarak yapıştırmayın; notebook paylaşıldığında sızar.

In [ ]:
import os

from huggingface_hub import login, whoami

token = None

# Colab Secrets
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    pass

# Kaggle Secrets (Add-ons > Secrets > HF_TOKEN)
if not token:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if not token:
    token = os.environ.get("HF_TOKEN")

if not token:
    from getpass import getpass

    token = getpass("HF token: ")

login(token=token)
print("giris:", whoami()["name"])

## 3. Temel modeli 4-bit yükle (QLoRA'nın "Q"su)

2B model fp16 olarak yaklaşık 5 GB yer kaplar; eğitim sırasında gradyanlar ve optimizer durumu eklendiğinde T4'ü zorlar. Bu yüzden model **NF4** biçiminde 4-bit yüklenir, hesaplar 16-bit yapılır. Ağırlıklar donuk kalır; yalnızca ekleyeceğimiz LoRA katmanları eğitilir.

`use_cache = False`: eğitim sırasında KV cache gradient checkpointing ile çakışır, kapatılır. Eval üretiminde geçici olarak açılır.

T4 kartı bf16 desteklemez; kod otomatik olarak fp16'ya düşer.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "google/gemma-2b-it"  # Gemma 1 — gemma-2-2b-it DEGIL

# torch.cuda.is_bf16_supported() T4'te de True donebiliyor (emulasyon sayiyor).
# Gercek donanim destegi Ampere ile, yani compute capability 8.0 ile baslar.
MAJOR, MINOR = torch.cuda.get_device_capability()
USE_BF16 = MAJOR >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("gpu:", torch.cuda.get_device_name(0), f"| compute capability: {MAJOR}.{MINOR}")
print("compute dtype:", COMPUTE_DTYPE, "(T4 icin float16 beklenir)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)
model.config.use_cache = False

print(model.config.model_type, "| katman:", model.config.num_hidden_layers, "| hidden:", model.config.hidden_size)
# Beklenen: gemma | katman: 18 | hidden: 2048

## 4. Dataset: yükle ve dengele

Veri seti `ayse-solmaz/gemma-2b-tr-dataset` içindeki `train.jsonl` dosyasıdır. Her satır `instruction` ve `output` alanları taşır.

**Neden filtre gerekiyor?** Veri setinin yarıya yakını soru-cevap değil, sohbet/motivasyon cümlesi ("Bugün kahvemi içerken..."). Bunların tamamıyla eğitilirse model kısa ve olgusal cevap yerine şiirsel muhabbet üretmeyi öğrenir. Hedef davranış "Türkiye'nin başkenti → Ankara" olduğu için:

- Soru-cevap satırlarının **tamamı** tutulur.
- Sohbet/motivasyon satırları **%15'e** indirilir (`CHATTY_KEEP_RATIO`). Doğal bir üslup kalsın diye tamamen atılmaz.

> **1. turdan ders:** Bu oran 0.4 iken model "Dışarıda bulunan başkent İstanbul'dur." gibi çağrışımsal cümleler üretmeye başladı. Sohbet satırları modele "konuyla ilgili yazmaya devam et" davranışını öğrettiği için oran 0.15'e indirildi.

Ayrıca küçük bir doğrulama kümesi ayrılır; `eval_loss` düşerken yükselmeye başlarsa ezberleme (overfitting) başlamış demektir.

In [ ]:
import re

from datasets import load_dataset

DATASET_REPO = "ayse-solmaz/gemma-2b-tr-dataset"
CHATTY_KEEP_RATIO = 0.15  # 1. tur 0.4 ile geveze cikti — dusuruldu
SEED = 42

raw = load_dataset(DATASET_REPO, data_files="train.jsonl", split="train")
# Yerel dosyayla calismak icin:
# raw = load_dataset("json", data_files="train.jsonl", split="train")

print("ham satir:", len(raw))

QA_PATTERN = re.compile(r"\?|nedir|neden|nasıl|hangi|kimdir|kaç|ne zaman|nerede", re.IGNORECASE)


def is_qa(example):
    return bool(QA_PATTERN.search(example["instruction"]))


qa_rows = raw.filter(is_qa)
chatty_rows = raw.filter(lambda ex: not is_qa(ex))

keep_n = int(len(chatty_rows) * CHATTY_KEEP_RATIO)
chatty_kept = chatty_rows.shuffle(seed=SEED).select(range(keep_n))

from datasets import concatenate_datasets

dataset = concatenate_datasets([qa_rows, chatty_kept]).shuffle(seed=SEED)

print(f"soru-cevap: {len(qa_rows)} | sohbet: {len(chatty_rows)} -> {keep_n} tutuldu")
print("egitim seti:", len(dataset))
print(dataset[0])

## 5. Gemma sohbet şablonuna çevir

Model, eğitimde gördüğü biçimi çıkarımda da bekler. Gemma Instruct şablonu şudur:

```
<start_of_turn>user
SORU<end_of_turn>
<start_of_turn>model
CEVAP<end_of_turn>
```

Bu, projedeki MLC yapılandırmasındaki `gemma_instruction` şablonuyla aynıdır. Farklı bir biçimde eğitirseniz model MLC'de servis edilirken beklenenden kötü davranır.

Gemma'nın güvenilir bir `system` rolü yoktur; talimat doğrudan kullanıcı turuna yazılır.

`MAX_LEN = 512`: veri setindeki en uzun örnekler bile bunun altında; daha büyük değer sadece bellek ve süre maliyeti getirir.

### Completion-only loss (1. turdan ders)

İlk turda model hem soruyu hem cevabı üretmeyi öğrendi ve talimat takibi zayıf kaldı. Bu turda **loss yalnızca cevap kısmından** alınıyor: soru token'larının etiketi `-100` yapılarak kayıp hesabından çıkarılır.

Ayrıca her örnek `<end_of_turn>` ile bitirilir; model böylece **durmayı** öğrenir. İlk turda cevaplar bitmeyip içine `model` kelimesi sızıyordu.

In [ ]:
MAX_LEN = 512
END_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")
print("end_of_turn id:", END_ID)


def build_example(example):
    user = example["instruction"].strip()
    extra = (example.get("input") or "").strip()
    if extra:
        user = f"{user}\n\n{extra}"
    answer = example["output"].strip()

    prompt = f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"
    full = f"{prompt}{answer}<end_of_turn>\n"

    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    input_ids = tokenizer(full, add_special_tokens=True)["input_ids"][:MAX_LEN]

    # Loss yalnizca cevaptan alinir: soru token'lari -100 ile maskelenir.
    labels = list(input_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


tokenized = dataset.map(build_example, remove_columns=dataset.column_names)

# Maskeleme dogru mu: yalnizca cevap kismi gorunmeli
sample = tokenized[0]
visible = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
print("\n--- ogrenilen kisim ---")
print(tokenizer.decode(visible))

split = tokenized.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

print("\ntrain:", len(train_ds), "| eval:", len(eval_ds))

## 6. LoRA yapılandırması

Tüm modeli değil, dikkat katmanlarına eklenen küçük düşük ranklı matrisleri eğitiyoruz. Eğitilebilir parametre oranı %1'in altında kalır; çıktı birkaç on MB'lık bir adaptördür.

| Parametre | Değer | Neden |
|---|---|---|
| `r` | 16 | Kapasite/bellek dengesi. 8 daha hafif ama daha az öğrenir; 32 küçük veri setinde ezber riskini artırır. |
| `lora_alpha` | 32 | Genel kural `alpha = 2 * r`. |
| `lora_dropout` | 0.05 | Küçük veri setinde ezberi frenler. |
| `target_modules` | q/k/v/o_proj | Dikkat projeksiyonları. MLP katmanları da eklenebilir ama bellek ve süre artar. |
| `task_type` | CAUSAL_LM | Metin üretimi görevi. |

`prepare_model_for_kbit_training` 4-bit modeli eğitime hazırlar: norm katmanlarını fp32'ye alır ve gradient checkpointing için giriş gradyanlarını etkinleştirir.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Eval soruları ve eğitim öncesi ölçüm

Her epoch sonunda aynı üç soru sorulur. Amaç kayıp (loss) grafiğine değil, **davranışa** bakmak:

| Soru | Beklenen | Neyi ölçer |
|---|---|---|
| Türkiye'nin başkenti neresidir? | Ankara | Türkçe olgusal doğruluk (eğitim öncesi bu soruda model "İstanbul" diyordu) |
| 2+2 kaç eder? | 4 | Basit akıl yürütmenin bozulmadığı |
| Su kaç derecede kaynar? | 100 °C | Genel bilginin korunduğu |

**Çıktıları yorumlama ipuçları**

- Cevap doğru ama uzun ve dolambaçlıysa: veri setindeki sohbet oranı yüksek — `CHATTY_KEEP_RATIO` düşürün.
- Cevap İngilizce geliyorsa: Türkçe örnek sayısı yetersiz veya epoch az.
- Cevap doğru ama aynı cümleyi tekrarlıyorsa: fazla epoch, ezber başlamış — epoch'u azaltın veya erken duran checkpoint'i kullanın.
- Üç soru da eğitim öncesiyle aynı kaldıysa: learning rate düşük ya da LoRA katmanları yanlış modüllere bağlanmış.
- Eğitim öncesi doğru olan bir cevap sonradan bozulduysa: veri kalitesi sorunlu (o konuda yanlış örnek var) ya da öğrenme oranı yüksek.

Aşağıdaki hücre eğitim **başlamadan önce** temel çizgiyi (baseline) kaydeder; karşılaştırma bunun üzerinden yapılır.

In [ ]:
EVAL_QUESTIONS = [
    "Türkiye'nin başkenti neresidir?",
    "2+2 kaç eder?",
    "Su kaç derecede kaynar?",
]


def ask(question, max_new_tokens=48):
    prompt = f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    was_training = model.training
    model.eval()
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # <end_of_turn> de durdurucu sayilmali; yoksa model konusmaya devam eder
            eos_token_id=[tokenizer.eos_token_id, END_ID],
            pad_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False
    if was_training:
        model.train()

    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def run_eval(title):
    print(f"\n===== {title} =====")
    for question in EVAL_QUESTIONS:
        print(f"S: {question}")
        print(f"C: {ask(question)}\n")


baseline = {q: ask(q) for q in EVAL_QUESTIONS}
run_eval("EGITIM ONCESI (baseline)")

## 8. Eğitim döngüsü

**Önerilen hiperparametreler (T4, ~3–4k satır):**

| Parametre | Öneri | Açıklama |
|---|---|---|
| `num_train_epochs` | **3** | 1. turda `eval_loss` 2. epoch'ta dibi gördü, 3–4'te yükseldi (ezber). `load_best_model_at_end` en iyi checkpoint'i geri yükler. |
| `per_device_train_batch_size` | **2** | T4 belleği için güvenli. Bellek hatası alırsanız 1'e düşürün. |
| `gradient_accumulation_steps` | **8** | Etkin batch = 2 × 8 = **16**. Daha kararlı gradyan. |
| `learning_rate` | **1e-4** | 1. turda 2e-4 ile model üslubu fazla bozdu. Yarıya indirildi. |
| `warmup_ratio` | **0.03** | İlk adımlarda oranı yumuşakça yükseltir. |
| `lr_scheduler_type` | **cosine** | Oranı eğitim sonuna doğru düşürür. |
| `logging_steps` | **20** | Loss'un düzenli görünmesi. |
| `eval_strategy` | **epoch** | Her epoch sonunda `eval_loss`. |
| `save_strategy` | **epoch** | En iyi checkpoint'i saklamak için. |

`EpochEvalCallback` her epoch sonunda üç soruyu tekrar sorar. Eğitimin ortasında "Ankara" doğru çıktıysa o checkpoint'i not alın; sonraki epoch'larda bozulursa en iyi checkpoint'e dönersiniz.

T4 üzerinde ~3k satır, 3 epoch yaklaşık 40–60 dakika sürer. Colab boşta bırakılırsa bağlantıyı keser ve **çekirdek sıfırlanınca bellekteki model kaybolur**; bu yüzden checkpoint'ler Google Drive'a yazılıyor. Sekmeyi kapatmayın, ara sıra pencereye tıklayın.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, TrainerCallback

# Colab kopunca checkpoint'ler kaybolmasin: varsa Drive'a yaz
OUTPUT_DIR = "./gemma-2b-it-tr-lora"
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/gemma-2b-it-tr-lora"
except Exception as exc:
    print("Drive baglanmadi, yerel diske yazilacak:", exc)
print("checkpoint dizini:", OUTPUT_DIR)


class EpochEvalCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch) if state.epoch is not None else "?"
        run_eval(f"EPOCH {epoch} SONU")


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    optim="paged_adamw_8bit",
    report_to="none",
    save_total_limit=2,
)

# Seq2Seq collator, -100 maskeli labels'i koruyarak pad eder.
# DataCollatorForLanguageModeling labels'i input_ids'ten yeniden uretir ve maskeyi siler.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[EpochEvalCallback()],
)

trainer.train()
print("egitim bitti")

## 9. Eğitim sonu kontrol

Aynı üç soruyu eğitim sonrası tekrar soruyoruz. Baseline ile yan yana bakın:

- **Başkent:** "İstanbul" → "Ankara" olduysa fine-tune hedefinin ana kısmı tutmuştur.
- **2+2:** "4" kalıyorsa akıl yürütme bozulmamıştır.
- **Kaynama:** "100" kalıyorsa genel bilgi korunmuştur.

Üçü de kötüyse: veri kalitesine, filtre oranına veya öğrenme oranına bakın; MLC'ye geçmeyin. Yalnızca başkent düzeldi, diğerleri bozulduysa epoch sayısı fazla olabilir — önceki epoch çıktısına dönün.

In [ ]:
run_eval("EGITIM SONRASI")

print("\n===== KARSILASTIRMA =====")
for question in EVAL_QUESTIONS:
    print(f"S: {question}")
    print(f"  once : {baseline[question]}")
    print(f"  sonra: {ask(question)}")
    print()

## 10. Adaptörü Hugging Face Hub'a yükle (private)

Kaydedilenler:

- LoRA adaptör ağırlıkları (`adapter_model.safetensors`)
- Adaptör yapılandırması (`adapter_config.json`)
- Tokenizer ve tokenizer config

**Önce private yükleyin.** Eval sonuçları iyi görünene kadar public yapmayın. Merge + MLC dönüşümü sonrası (Faz 5) public açmak güvenlidir.

`HUB_REPO` içindeki kullanıcı adını kendi HF kullanıcı adınızla değiştirin. `whoami()` çıktısı doğruysa bir satır aşağıda otomatik kurulur.

Bu hücre yalnızca adaptörü yükler. Temel modele birleştirme (merge) ve MLC `convert_weight` ayrı adımlardır; Windows native MLC kırık olduğu için convert Colab/Linux'ta yapılmalıdır.

In [ ]:
from huggingface_hub import whoami

hf_user = whoami()["name"]
HUB_REPO = f"{hf_user}/gemma-2b-it-tr-lora"  # ornek: ayse-solmaz/gemma-2b-it-tr-lora

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HUB_REPO, private=True)
tokenizer.push_to_hub(HUB_REPO, private=True)

print(f"yuklendi (private): https://huggingface.co/{HUB_REPO}")
print("icerik: adapter + tokenizer + config")
print("sonraki adim: Colab'da merge + MLC convert (Faz 5), sonra volume swap")

## 11. Kurtarma: checkpoint'ten yükle ve değerlendir

Oturum koptuğunda, çekirdek yeniden başladığında veya kütüphane hatası aldığınızda bellekteki model kaybolur ama **diskteki checkpoint'ler durur** (`./gemma-2b-it-tr-lora/checkpoint-*`). Baştan eğitmeye gerek yoktur.

Sıra önemlidir:

1. Önce checkpoint'i Hugging Face'e yedekleyin. `huggingface_hub` tek başına çalışır, `transformers` bozuk olsa bile yükleme yapılır. Oturum kapanırsa `/kaggle/working` silinir.
2. Sonra ortamı sabit sürümlerle kurup çekirdeği yeniden başlatın.
3. En son bu bölümdeki hücreyle temel modeli + adaptörü yükleyip değerlendirin.

Bu bölüm bölüm 9'un (eğitim sonu kontrol) yerini tutar; farkı, eğitim oturumuna bağlı olmaması.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "google/gemma-2b-it"
ADAPTER_PATH = "gemma-2b-it-tr-lora/checkpoint-519"  # veya HF repo id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
model.config.use_cache = True

END_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")


def ask(question, max_new_tokens=48):
    prompt = f"<start_of_turn>user\n{question}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=[tokenizer.eos_token_id, END_ID],
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


CHECK_QUESTIONS = [
    "Türkiye'nin başkenti neresidir?",
    "2+2 kaç eder?",
    "Su kaç derecede kaynar?",
    "Bu projenin backend dili nedir?",
    "Access token kaç dakika geçerlidir?",
    "Merhaba, nasılsın?",
]

for question in CHECK_QUESTIONS:
    print(f"S: {question}")
    print(f"C: {ask(question)}\n")